# Day 5 — Deploy Mistral as a SageMaker Endpoint

Deploys **base Mistral-7B-Instruct-v0.3** as a SageMaker inference endpoint. To deploy your own fine-tuned model instead, see Step 3 (Option B).

**Cost:** ~$1.50/hr while the endpoint runs. Run the teardown cell at the bottom when finished.

**Prereqs:**
- Quota for `ml.g5.2xlarge for endpoint usage` approved (Service Quotas → SageMaker)
- HuggingFace token from https://huggingface.co/settings/tokens

## Step 1 — Setup

In [ ]:
import json
import sagemaker
import boto3
from sagemaker.huggingface import HuggingFaceModel, get_huggingface_llm_image_uri

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name

print(f"Role:   {role}")
print(f"Region: {region}")

## Step 2 — Clean up any stale endpoints from previous attempts

If you've iterated on this notebook, old endpoint configs may block reusing names.

In [ ]:
sm = boto3.client("sagemaker")

for name in ["mistral-base-v1", "mistral-ft-v1", "mistral-ft-v2"]:
    for fn, key in [(sm.delete_endpoint, "EndpointName"),
                    (sm.delete_endpoint_config, "EndpointConfigName")]:
        try:
            fn(**{key: name})
            print(f"✅ Deleted {key}: {name}")
        except Exception:
            pass
print("Cleanup done")

## Step 3 — Configure the model

**Option A (default):** Deploy base Mistral-7B-Instruct-v0.3 directly from HuggingFace. Works immediately.

**Option B (advanced):** Deploy your own fine-tuned model from S3. Tarball MUST be in safetensors format — TGI 2.0 cannot deploy `.bin` files.

Pick one. Comment out the other.

In [ ]:
# ============================================================
# OPTION A — Base Mistral (default; works without an S3 model)
# ============================================================
HF_TOKEN = "<<PASTE_YOUR_HF_TOKEN_HERE>>"

config = {
    "HF_MODEL_ID": "mistralai/Mistral-7B-Instruct-v0.3",
    "HF_TOKEN": HF_TOKEN,
    "SM_NUM_GPUS": json.dumps(1),
    "MAX_INPUT_LENGTH": json.dumps(2048),
    "MAX_TOTAL_TOKENS": json.dumps(4096),
    "MAX_BATCH_PREFILL_TOKENS": json.dumps(4096),
}
model_data = None
endpoint_name = "mistral-base-v1"

# ============================================================
# OPTION B — Your fine-tuned model (uncomment to use)
# ============================================================
# model_s3_uri = "<<PASTE_YOUR_SAFETENSORS_S3_URI_HERE>>"  # e.g. s3://bucket/path/model.tar.gz
# config = {
#     "HF_MODEL_ID": "/opt/ml/model",
#     "SM_NUM_GPUS": json.dumps(1),
#     "MAX_INPUT_LENGTH": json.dumps(2048),
#     "MAX_TOTAL_TOKENS": json.dumps(4096),
#     "MAX_BATCH_PREFILL_TOKENS": json.dumps(4096),
# }
# model_data = model_s3_uri
# endpoint_name = "mistral-ft-v1"

assert HF_TOKEN.startswith("hf_") or model_data is not None, \
    "Either paste your HF token or uncomment Option B with an S3 URI"
print(f"Endpoint name: {endpoint_name}")
print(f"Model: {config.get('HF_MODEL_ID') if model_data is None else model_data}")

## Step 4 — Get the TGI inference image

In [ ]:
try:
    llm_image = get_huggingface_llm_image_uri("huggingface", version="2.0.0")
except Exception:
    llm_image = (
        f"763104351884.dkr.ecr.{region}.amazonaws.com/"
        "huggingface-pytorch-tgi-inference:"
        "2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
    )

print(f"TGI image: {llm_image}")

## Step 5 — 🔥 DEPLOY

⚠️ **This cell starts costing ~$1.50/hr** when the endpoint reaches `InService`.

Takes ~10 minutes.

In [ ]:
if model_data is None:
    # Option A: base Mistral, no model_data
    llm_model = HuggingFaceModel(
        role=role,
        image_uri=llm_image,
        env=config,
    )
else:
    # Option B: your fine-tuned tarball
    llm_model = HuggingFaceModel(
        model_data=model_data,
        role=role,
        image_uri=llm_image,
        env=config,
    )

llm = llm_model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1,
    instance_type="ml.g5.2xlarge",
    container_startup_health_check_timeout=600,
)

print(f"✅ Endpoint '{endpoint_name}' is live")

## Step 6 — Sanity test

Mistral-7B-Instruct uses `<s>[INST] ... [/INST]` as its trained prompt format. Wrap accordingly.

In [ ]:
import json, boto3
runtime = boto3.client("sagemaker-runtime")

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps({
        "inputs": "<s>[INST] What is deep learning? [/INST]",
        "parameters": {
            "max_new_tokens": 300,
            "do_sample": True,
            "temperature": 0.7,
            "return_full_text": False,
        },
    }),
)
result = json.loads(response["Body"].read().decode())
print(result[0]["generated_text"])

## Step 7 — Save the endpoint name for Lambda

Copy the value below and set it as the `ENDPOINT_NAME` environment variable on your Lambda function.

In [ ]:
print("=" * 60)
print("PASTE INTO LAMBDA → Configuration → Environment variables:")
print("=" * 60)
print(f"  Key:   ENDPOINT_NAME")
print(f"  Value: {endpoint_name}")
print("=" * 60)

## 🚨 TEARDOWN — Run this when you're done

Each hour the endpoint is up = ~$1.50.

In [ ]:
sm = boto3.client("sagemaker")

try:
    sm.delete_endpoint(EndpointName=endpoint_name)
    sm.delete_endpoint_config(EndpointConfigName=endpoint_name)
    sm.delete_model(ModelName=llm_model.name)
    print(f"✅ Deleted endpoint, config, and model for '{endpoint_name}'")
except Exception as e:
    print(f"Cleanup note (likely already deleted): {e}")

print("\nVerify in console: SageMaker → Inference → Endpoints — should be empty")